# Optimizing with `popcorn.compile` — experimental

The [previous notebook](./01_optimize_modeling_code.ipynb) called popcorn kernels from the modeling code. This one injects them into modeling code that never imports popcorn: `popcorn.compile.enable()` traces a replacement pattern from every kernel's reference and installs an inductor pass, so `torch.compile` rewrites matched subgraphs to dispatched popcorn ops — forward and backward graphs alike.

We reuse [`modeling_butter.py`](./modeling_butter.py) in its attention-only configuration with `dispatch=False`, so every op runs as its plain-torch reference — a stand-in for a model you cannot edit. The delta-rule layers sit this one out: a python scan over the sequence never produces a matchable graph (see the [docs](https://tilde-research.github.io/popcorn/docs/torch-compile) for this and other limits) — for scan-style kernels, call them directly as in the first notebook.

**This integration is experimental**: pattern matching is best-effort, and a rewrite is not automatically a win — inductor's own fused codegen is at roofline for many small memory-bound ops. Measure end to end.

In [1]:
import torch

from modeling_butter import ButterConfig, ButterForCausalLM

torch.manual_seed(0)
device, dtype = "cuda", torch.float32  # float32: see the dtype note at the end

config = ButterConfig(layer_types=("attn",), vocab_size=131072, dispatch=False)  # plain-torch references only
model = ButterForCausalLM(config).to(device, dtype)

input_ids = torch.randint(config.vocab_size, (16, 1024), device=device)
labels = torch.randint(config.vocab_size, (16, 1024), device=device)
batch = {"input_ids": input_ids, "labels": labels}


def train_step(**batch):
    loss = model(**batch)
    loss.backward()
    model.zero_grad(set_to_none=True)
    return loss.detach()

## Baseline: plain `torch.compile`

Compile the training step as-is and profile it against eager with popcorn's harness (same `compare` as the kernel test suite: `do_bench` timing, peak memory, outputs graded against a float64 run).

In [2]:
from popcorn.bench import compare


def report(result, what):
    bench = result.bench
    speed = bench["ref_fwd_ms"] / bench["fwd_ms"]
    print(f"{what:24s} {bench['fwd_ms']:6.1f} ms vs eager {bench['ref_fwd_ms']:6.1f} ms   ({speed:.2f}x)")
    if "fwd_mem_mb" in bench:
        print(f"{'':24s} peak {bench['fwd_mem_mb'] / 1024:5.2f} GB vs eager {bench['ref_fwd_mem_mb'] / 1024:5.2f} GB")
    print(f"{'':24s} loss deviation {result.fwd['out0'].err:.2e} on scale {result.fwd['out0'].scale:.1f}")


compiled = torch.compile(train_step)
plain = compare(compiled, train_step, batch, backward=False, repeats=3)
report(plain, "torch.compile")

torch.compile             942.7 ms vs eager  980.7 ms   (1.04x)
                         peak 18.63 GB vs eager 36.13 GB
                         loss deviation 9.54e-07 on scale 12.2


## Enable popcorn inside the compiler

`enable(ops=...)` traces the replacement patterns (a few seconds per op, cached for the process) and installs the pass. Everything compiled afterwards gets rewritten wherever a pattern matches and a non-torch backend is expected to win; `torch._dynamo.reset()` forces recompilation of what we already compiled.

We scope it to the one op with an algorithmic edge here: at a 131k vocabulary the reference loss materializes an 8 GB logits matrix (plus its gradient and softmax intermediates), which the fused backends never allocate. Norm and swiglu patterns are available too, but inductor's own codegen already serves those well — rewriting them just adds custom-op boundaries.

In [3]:
import popcorn.compile

patterns = popcorn.compile.enable(ops=["linear_cross_entropy"])
torch._dynamo.reset()
print(f"{patterns} replacement patterns registered")

with_popcorn = compare(compiled, train_step, batch, backward=False, repeats=3)
report(with_popcorn, "torch.compile + popcorn")
print(f"\ncompiled-vs-compiled: {plain.bench['fwd_ms'] / with_popcorn.bench['fwd_ms']:.2f}x time, "
      f"{plain.bench['fwd_mem_mb'] / with_popcorn.bench['fwd_mem_mb']:.1f}x memory")

89 replacement patterns registered


torch.compile + popcorn  1477.3 ms vs eager  983.9 ms   (0.67x)
                         peak 12.75 GB vs eager 36.13 GB
                         loss deviation 9.54e-07 on scale 12.2

compiled-vs-compiled: 0.64x time, 1.5x memory


## Reading the result

Peak memory walks down in two steps — eager 36 GB, compiled 19 GB, popcorn 13 GB — and the last step is the fused loss: the reference materializes an 8 GB logits matrix inside the compiled graph, the rewritten graph never allocates it.

The step gets slower, though, and that trade is structural. Popcorn's training binding does not thread saved activations across the pattern boundary; its backward op **replays the forward** through the dispatcher — gradient-checkpoint semantics, one extra forward per backward. So a training rewrite buys memory with time unless the backend is decisively faster than inductor's codegen, which fla's float32 fused loss here is not (it is at parity). Inference graphs have no backward and skip the replay: there, a rewrite is a plain speed contest between the backend and inductor.

**Why float32?** A rewrite only goes in when a non-torch backend is expected to serve the call. The fused loss backends return the loss in float32, so under bfloat16 they fail popcorn's dtype round-trip contract, the dispatcher would fall back to torch, and the pass — knowing that — declines the rewrite. Run this notebook in bfloat16 and the loss keeps its logits matrix; that is the eligibility gate working, not a missed match.

Scope `enable(ops=[...])` to kernels whose trade you want, measure end to end, and treat the whole pass as what it is — experimental. `popcorn.compile.disable()` uninstalls it. When you control the modeling code, calling the kernels directly (first notebook) gets the speed without the replay: eager popcorn keeps each backend's own autograd.